In [1]:
from pathlib import Path
from datetime import datetime
from typing import List
from types import SimpleNamespace
import pickle
import numpy as np
import torch
from pymatgen.core import Structure, Lattice, Species, Element


from chggen.pl_data.dataset import CHGNetDataset
from chggen.pl_modules.model_egnn import CHGGen
from chggen.common.data_utils import get_scaler_from_data_list
from chggen.pl_data.datamodule import CrystDataModule


from torch_geometric.data import Data
from torch import nn
import torch.nn.functional as F

def get_scaler(dataset, use_prop_scaler = False, 
               scaler_path = None):
    # Load once to compute property scaler
    if scaler_path is None:
        lattice_scaler = get_scaler_from_data_list(
            dataset.cached_data,
            key='scaled_lattice')
        if use_prop_scaler:
            NotImplementedError("Not implemented the multi prop scaler yet.")
    else:
        lattice_scaler = torch.load(
            Path(scaler_path) / 'lattice_scaler.pt')
    return lattice_scaler



/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

dataset = CHGNetDataset(path= '/home/zhongpc/chggen/data/perov_5/test_zpc.csv',
name = 'zpc_debug',
prop_list = ['heat_all'],
)

lattice_scaler = get_scaler(dataset= dataset)
data_list = [dataset[i] for i in range(len(dataset))]
datamodule = CrystDataModule(train_dataset= dataset,
                             val_dataset= dataset,
                             num_workers=8,
                             batch_size= 32,
                                )


model_hparams ={'latent_dim': 64, 'hidden_dim': 128, 
                'predict_property': True, 'property_dim': 1, # predict the multiple property 
                'load_pretrain': True, 'fc_num_layers': 1, 
                'sigma_F_begin': 10.0, 'sigma_F_end': 0.01, 
                'sigma_L_begin': 1.0, 'sigma_L_end': 0.01, 
                'type_sigma_begin': 5.0, 'type_sigma_end': 0.01,
                'max_atoms': 20, # should be larger than the training set.
                'num_noise_level': 1, 
                'lattice_scale_method': 'scale_length', 
                'cost_natom': 1.0, 'cost_coord': 10.0, 'cost_type': 1.0, 'cost_lattice': 10.0, 'cost_composition': 1.0, 'cost_edge': 10.0, 'cost_property': 1.0,
                'beta': 0.01,
                'teacher_forcing_lattice': True,
                'teacher_forcing_max_epoch': 1000,
                'decoder': 'egnn'}

chggen = CHGGen(lattice_scaler= lattice_scaler, 
                hparams_dict= model_hparams)



device = torch.device('cpu')
chggen.to(device = device)


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 291.89it/s]


CHGNet initialized with 400,438 parameters
CHGNet initialized with 400,438 parameters


/home/zhongpc/chggen/chggen/common/data_utils.py:647: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:245.)
  targets = torch.tensor([d[key] for d in data_list])
/home/zhongpc/chggen/chggen/common/data_utils.py:615: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)


CHGGen(
  (encoder): CHGNet_encoder(
    (composition_model): AtomRef(
      (fc): Linear(in_features=94, out_features=1, bias=False)
    )
    (graph_converter): CrystalGraphConverter(algorithm='fast', atom_graph_cutoff=5, bond_graph_cutoff=3)
    (atom_embedding): AtomEmbedding(
      (embedding): Embedding(94, 64)
    )
    (bond_basis_expansion): BondEncoder(
      (rbf_expansion_ag): RadialBessel(
        (smooth_cutoff): CutoffPolynomial()
      )
      (rbf_expansion_bg): RadialBessel(
        (smooth_cutoff): CutoffPolynomial()
      )
    )
    (bond_embedding): Linear(in_features=9, out_features=64, bias=False)
    (bond_weights_ag): Linear(in_features=9, out_features=64, bias=False)
    (bond_weights_bg): Linear(in_features=9, out_features=64, bias=False)
    (angle_basis_expansion): AngleEncoder(
      (fourier_expansion): Fourier()
    )
    (angle_embedding): Linear(in_features=9, out_features=64, bias=False)
    (atom_conv_layers): ModuleList(
      (0-3): 4 x AtomConv(


In [3]:

# langevin dynamics
ld_kwargs = SimpleNamespace(n_step_each = 10,
                            step_lr = 1e-4,
                            min_sigma = 0,
                            save_traj = False,
                            disable_bar = False,
                            compute_force = True,
                            beta_c = 0, # property update rate
                            beta_f = 0, # atomic force update rate
                            )

z = torch.rand(1, 64, requires_grad= True, device = device)
results = chggen.langevin_dynamics_guidance(z = z, 
                                            prop_guidance = torch.tensor(-0.05, device= device), 
                                            ld_kwargs= ld_kwargs)


/home/zhongpc/chggen/chggen/common/data_utils.py:625: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)
  0%|                                                                                                                            | 0/1 [00:00<?, ?it/s]/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/core/periodic_table.py:221: UserWarning: No Pauling electronegativity for He. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  warnings.warn(


max lattice:  tensor(0.9738)
FrRaAcYbDyTmNpPaVFeCuAgHeKrCl
max lattice:  tensor(6.0191, grad_fn=<MaxBackward1>)
FrRaAcYbDyTmNpPaVFeCuAgHeKrCl
max lattice:  tensor(6.0241, grad_fn=<MaxBackward1>)
FrRaAcYbDyTmNpPaVFeCuAgHeKrCl
max lattice:  tensor(6.0175, grad_fn=<MaxBackward1>)
FrRaAcYbDyTmNpPaVFeCuAgHeKrCl
max lattice:  tensor(6.0196, grad_fn=<MaxBackward1>)
FrRaAcYbDyTmNpPaVFeCuAgHeKrCl
max lattice:  tensor(6.0453, grad_fn=<MaxBackward1>)
FrRaAcYbDyTmNpPaVFeCuAgHeKrCl
max lattice:  tensor(6.0179, grad_fn=<MaxBackward1>)
FrRaAcYbDyTmNpPaVFeCuAgHeKrCl
max lattice:  tensor(6.0241, grad_fn=<MaxBackward1>)
FrRaAcYbDyTmNpPaVFeCuAgHeKrCl
max lattice:  tensor(6.0135, grad_fn=<MaxBackward1>)
FrRaAcYbDyTmNpPaVFeCuAgHeKrCl


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.08s/it]

max lattice:  tensor(6.0020, grad_fn=<MaxBackward1>)
FrRaAcYbDyTmNpPaVFeCuAgHeKrCl


In [4]:

# save the results from langevin dynamics

lattices = results['lattices']
num_atoms = results['num_atoms']
frac_coords = results['frac_coords']
atom_types = results['atom_types']

batch = torch.arange(len(num_atoms))
batch = batch.repeat_interleave(num_atoms)


In [5]:
lattices.shape

torch.Size([15, 3, 3])

In [6]:
num_atoms

tensor([15])

In [7]:
!mkdir ./test_models/EGNN_structures/
for ii in range(len(num_atoms)):
    indices = torch.where(batch == ii)[0]
    print(ii, indices)
    if len(indices) == 0:
        continue
    

    Latt = Lattice(lattices[ii].cpu().detach().numpy())
    frac_ = frac_coords[indices]
    type_ = atom_types[indices]
    species_ = [Element.from_Z(ele_Z) for ele_Z in type_]
    
    s_gen = Structure(lattice= Latt , species= species_, coords= frac_.detach().numpy(),
                      to_unit_cell=False,coords_are_cartesian=False);
    print(s_gen.composition)
    s_gen.to(filename= './test_models/EGNN_structures/prop_guidance_' + str(ii) + '.cif')
    


print("Done")


mkdir: cannot create directory ‘./test_models/EGNN_structures/’: File exists
0 tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14])
Cu1 Ag1 Fr1 Tm1 Np1 V1 He1 Yb1 Dy1 Cl1 Pa1 Ra1 Fe1 Ac1 Kr1
Done
